# 00 Data Compression

The raw AMEX training table contains 5.5 million monthly statements and is too large to explore comfortably on a 16 GB laptop. This notebook builds a reproducible storage layer before any analysis or modeling.

It has two stages:

1. stream the raw CSV into a typed, compressed float32 Parquet file
2. convert  continuous columns to float16 and write a Zstandard-compressed Feather file for fast downstream scans.


### Part I — Raw CSV to Parquet


In [3]:
from pathlib import Path

def find_project_root() -> Path:
    """Use the parent of the notebook working directory as the project root."""
    project_root = Path.cwd().resolve().parent
    return project_root

import json, os, time
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

PROJECT_ROOT = find_project_root()
INPUT = PROJECT_ROOT / 'data' / 'train_data.csv'
OUTPUT = PROJECT_ROOT / '00_compression' / 'train_data_compressed.parquet'
CHUNK_SIZE = 100_000
COMPRESSION = 'zstd'
COMPRESSION_LEVEL = 6

#### Define the schema and safe conversion rules

The anonymous AMEX columns are not all  equivalent. IDs, dates, binary values, categorical codes, and continuous measurements need different representations.
Categorical mappings are fixed globally, ensuring that the same category always receives the same integer code.


In [ ]:
CATEGORICAL_COLUMNS = [
    'B_30', 'B_38', 'D_114', 'D_116', 'D_117', 'D_120',
    'D_126', 'D_63', 'D_64', 'D_66', 'D_68'
]

STRING_CATEGORY_MAPS = {
    'D_63': {'CL': 0, 'CO': 1, 'CR': 2, 'XL': 3, 'XM': 4, 'XZ': 5},
    'D_64': {'-1': 0, 'O': 1, 'R': 2, 'U': 3},
}

def customer_suffix_to_int64(series: pd.Series) -> pd.Series:
    values = np.fromiter((int(x[-16:], 16) for x in series), dtype=np.uint64, count=len(series))
    return pd.Series(values.view(np.int64), index=series.index, name='customer_ID')

def encode_int8(series: pd.Series, name: str) -> pd.Series:
    if name in STRING_CATEGORY_MAPS:
        raw = series.astype('string')
        encoded = raw.map(STRING_CATEGORY_MAPS[name])
        return encoded.astype('Int8')
    numeric = pd.to_numeric(series, errors='coerce')
    return numeric.astype('Int8')

def restore_hex_suffix(value: int) -> str:
    return format(int(value) & ((1 << 64) - 1), '016x')

#### Stream the CSV into Parquet

The raw CSV is read in chunks. The first chunk establishes the Arrow schema, and every subsequent chunk must match it exactly.

In [ ]:
header = pd.read_csv(INPUT, nrows=0).columns.tolist()
required = {'customer_ID', 'S_2', 'B_31', *CATEGORICAL_COLUMNS}
missing = required.difference(header)
numeric_columns = [c for c in header if c not in {'customer_ID', 'S_2', 'B_31', *CATEGORICAL_COLUMNS}]

writer = None
rows_written = 0
unique_full_ids = set()
suffix_to_full_id = {}
sample_before = None
started = time.time()

try:
    for chunk_number, chunk in enumerate(pd.read_csv(INPUT, chunksize=CHUNK_SIZE, low_memory=False), start=1):
        original_ids = chunk['customer_ID'].astype(str)
        compact_ids = customer_suffix_to_int64(original_ids)

        for full_id, suffix in zip(original_ids.unique(), customer_suffix_to_int64(pd.Series(original_ids.unique()))):
            suffix = int(suffix)
            previous = suffix_to_full_id.setdefault(suffix, full_id)
            if previous != full_id:
                raise ValueError(f'customer_ID conflict: {previous} and {full_id}')
            unique_full_ids.add(full_id)

        if sample_before is None:
            sample_before = chunk.head(1000).copy()

        chunk['customer_ID'] = compact_ids
        chunk['S_2'] = pd.to_datetime(chunk['S_2'], format='%Y-%m-%d', errors='raise').dt.date
        for col in CATEGORICAL_COLUMNS:
            chunk[col] = encode_int8(chunk[col], col)
        chunk['B_31'] = encode_int8(chunk['B_31'], 'B_31')
        chunk[numeric_columns] = chunk[numeric_columns].astype('float32')

        table = pa.Table.from_pandas(chunk, preserve_index=False)
        if writer is None:
            metadata = dict(table.schema.metadata or {})
            metadata[b'category_mappings'] = json.dumps(STRING_CATEGORY_MAPS, sort_keys=True).encode()
            metadata[b'customer_ID_transform'] = b'last_16_hex_chars_reinterpreted_as_int64'
            table = table.replace_schema_metadata(metadata)
            writer = pq.ParquetWriter(
                OUTPUT, table.schema, compression=COMPRESSION,
                compression_level=COMPRESSION_LEVEL, use_dictionary=True
            )
        writer.write_table(table, row_group_size=CHUNK_SIZE)
        rows_written += len(chunk)
        elapsed = time.time() - started
        print(f'chunk {chunk_number:>3}: {rows_written:,} rows, {elapsed/60:.1f} minutes')
finally:
    if writer is not None:
        writer.close()

#### Validate the Parquet artifact

Validation checks the total row and column counts, column order, target dtypes, metadata, and file readability.

In [ ]:
parquet_file = pq.ParquetFile(OUTPUT)
assert parquet_file.metadata.num_rows == rows_written
assert parquet_file.metadata.num_columns == len(header)

schema = parquet_file.schema_arrow
assert schema.field('customer_ID').type == pa.int64()
assert schema.field('S_2').type == pa.date32()
for col in CATEGORICAL_COLUMNS + ['B_31']:
    assert schema.field(col).type == pa.int8(), (col, schema.field(col).type)
for col in numeric_columns:
    assert schema.field(col).type == pa.float32(), (col, schema.field(col).type)

sample_after = parquet_file.read_row_group(0).slice(0, len(sample_before)).to_pandas()
expected_suffixes = sample_before['customer_ID'].str[-16:].str.lower().tolist()
actual_suffixes = [restore_hex_suffix(x) for x in sample_after['customer_ID']]
assert actual_suffixes == expected_suffixes

before_numeric = sample_before[numeric_columns].astype('float64')
after_numeric = sample_after[numeric_columns].astype('float64')
max_abs_error = (before_numeric - after_numeric).abs().max().max()

input_gib = INPUT.stat().st_size / 2**30
output_gib = OUTPUT.stat().st_size / 2**30
print(f'Test passed：{parquet_file.metadata.num_rows:,} rows × {parquet_file.metadata.num_columns} columns')
print(f'Largest float32 absolute error（first 1,000 rows）: {max_abs_error:.3g}')
print(f'CSV:     {input_gib:.2f} GiB')
print(f'Parquet: {output_gib:.2f} GiB')
print(f'After compression size is original file\'s {output_gib/input_gib:.1%}, which saves {(1-output_gib/input_gib):.1%}')


### Part II — float32 Parquet to float16 Feather

The second stage optimizes for repeated local analysis. Feather provides fast Arrow IPC reads, while float16 further reduces the memory required for the many anonymous continuous variables. This is a more aggressive precision trade-off.


In [ ]:
from pathlib import Path
import os
import time
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.ipc as ipc
import pyarrow.parquet as pq

INPUT = PROJECT_ROOT / '00_compression' / 'train_data_compressed.parquet'
OUTPUT = PROJECT_ROOT / 'data' / 'train_data_float16_zstd.feather'
COMPRESSION = 'zstd'

#### Construct the destination schema

Only fields currently stored as float32 are changed to float16. Customer ID, statement date, categorical columns, and `B_31` retain their existing types.


In [ ]:
parquet_file = pq.ParquetFile(INPUT)
source_schema = parquet_file.schema_arrow
float_columns = [field.name for field in source_schema if pa.types.is_float32(field.type)]

target_fields = [
    pa.field(field.name, pa.float16(), nullable=field.nullable, metadata=field.metadata)
    if pa.types.is_float32(field.type) else field
    for field in source_schema
]
target_metadata = dict(source_schema.metadata or {})
target_metadata[b'float_conversion'] = b'float32_to_float16_lossy'
target_metadata[b'container'] = b'Feather_V2_Arrow_IPC_with_Zstandard'
target_schema = pa.schema(target_fields, metadata=target_metadata)


#### Convert one Parquet row group at a time


In [ ]:
FLOAT16_MAX = np.finfo(np.float16).max
write_options = ipc.IpcWriteOptions(compression=COMPRESSION)
rows_written = 0
started = time.time()
sample_before = None

with pa.OSFile(str(OUTPUT), 'wb') as sink:
    with ipc.new_file(sink, target_schema, options=write_options) as writer:
        for row_group_index in range(parquet_file.metadata.num_row_groups):
            table = parquet_file.read_row_group(row_group_index)
            if sample_before is None:
                sample_before = table.slice(0, 1000)

            converted_columns = []
            for field, column in zip(source_schema, table.columns):
                if pa.types.is_float32(field.type):
                    finite = column.filter(pa.compute.is_finite(column))
                    if len(finite):
                        maximum = pa.compute.max(pa.compute.abs(finite)).as_py()
                    combined = column.combine_chunks()
                    null_mask = combined.is_null().to_numpy(zero_copy_only=False)
                    values = combined.to_numpy(zero_copy_only=False).astype(np.float16)
                    column = pa.array(values, mask=null_mask, type=pa.float16())
                converted_columns.append(column)

            converted = pa.Table.from_arrays(converted_columns, schema=target_schema)
            writer.write_table(converted)
            rows_written += len(converted)
            elapsed = time.time() - started
            print(
                f'row group {row_group_index + 1:>2}/{parquet_file.metadata.num_row_groups}: '
                f'{rows_written:,} rows，{elapsed / 60:.1f} minutes'
            )


#### Validate the final Feather file


In [ ]:
with pa.memory_map(str(OUTPUT), 'r') as source:
    reader = ipc.open_file(source)
    output_schema = reader.schema
    output_rows = sum(reader.get_batch(i).num_rows for i in range(reader.num_record_batches))
    sample_after = reader.get_batch(0).slice(0, 1000)

assert output_rows == parquet_file.metadata.num_rows == rows_written
assert len(output_schema) == len(source_schema)
assert output_schema.names == source_schema.names

for before_field, after_field in zip(source_schema, output_schema):
    expected = pa.float16() if pa.types.is_float32(before_field.type) else before_field.type
    assert after_field.type == expected, (after_field.name, after_field.type, expected)

before_df = sample_before.select(float_columns).to_pandas().astype('float32')
after_df = pa.Table.from_batches([sample_after]).select(float_columns).to_pandas().astype('float32')
assert before_df.isna().equals(after_df.isna())
new_non_finite = np.isfinite(before_df.to_numpy()) & ~np.isfinite(after_df.to_numpy())

absolute_error = (before_df - after_df).abs()
nonzero = before_df.abs() > 0
relative_error = (absolute_error / before_df.abs()).where(nonzero)
max_absolute_error = absolute_error.max().max()
max_relative_error = relative_error.max().max()

input_gib = INPUT.stat().st_size / 2**30
output_gib = OUTPUT.stat().st_size / 2**30
original_csv = PROJECT_ROOT / 'data' / 'train_data.csv'

if original_csv.exists():
    csv_gib = original_csv.stat().st_size / 2**30
    print(f'Original CSV：{csv_gib:.2f} GiB；Saved：{1 - output_gib / csv_gib:.1%}')
